# Температурный мониторинг здания

Актуальная версия после ответа руководителя: °C, ±0,1 °C, разрешение 0,25 см,
UTC+3 и −300 как конец трассы. Нет требования 40 точек. Действующая постановка
в docs/task_revision.md. Notebook читает результаты готового pipeline, не подбирает
параметры на test. Сначала выполните scripts/run_solution.py из корня проекта.


In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display

root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
output = root / 'artifacts/final'
summary = json.loads((output / 'solution.json').read_text())
assert summary['schema_version'] == 'building-temperature-v2'
print(summary['status'], sum(f['profiles'] for f in summary['files']), 'profiles')
summary['semantics']


## Прогноз и вклад пространственных признаков

Каждый вариант выбран по validation; test-метрики показываются отдельно.


In [ ]:
for f in summary['files']:
    print(f['source_file'], f['selected_model'], f['test_metrics_selected'])
    for family, model in f['feature_ablation'].items():
        print(' ', family, 'validation:', f['validation_metrics'][model]['mae_c'],
              'test:', f['test_metrics'][model]['mae_c'])


In [ ]:
for f in summary['files']:
    print(f['source_file'])
    display(Image(filename=str(output / Path(f['source_file']).stem / 'diagnostics.png')))


## Кандидаты событий

Температура не доказывает причину протечки. Поля peak_evidence позволяют проверить исходные строки.


In [ ]:
for event in summary['events'][:10]:
    print(event['source_file'], event['kind'], event['start'], event['priority_score'])
    print(json.dumps(event['peak_evidence'], ensure_ascii=False, indent=2))


## Ограничения оценки

Набор записей уже исследован в v1. Повторная оценка v2 не является новым независимым тестом.
Нет внешних меток аварий и проверки на новом объекте. Число кандидатов и MAE не являются качеством обнаружения протечек.
Для физической локализации нужно проверить индексный шаг и карту трассы.
